# Clase 14 — De Regresión a Clasificación

**Diplomado en Data Science Aplicada con Python para la Toma de Decisiones**
Arca Continental Ecuador | UDLA
*10 de Abril, 2026*

---

## Objetivos de hoy

1. **P-values** de los coeficientes: ¿son reales o ruido?
2. **Selección de variables**: cuáles quitar y cómo verificar
3. **Supuestos** de la regresión lineal: cuándo NO funciona
4. **Clasificación binaria**: ¿y si el target es sí/no?
5. **LinearRegression con umbral** — funciona, pero...
6. **Regresión logística** — la sigmoide y `predict_proba`

---

## Bloque A — Cierre de regresión lineal (con insurance)

## 0. Imports y datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 5)

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (r2_score, mean_absolute_error, mean_squared_error,
                             accuracy_score, confusion_matrix, classification_report)

print("Listo.")

In [ ]:
# Cargamos el dataset de la clase 13
URL_INS = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"
df_ins = pd.read_csv(URL_INS)

# Encoding (mismo que clase 13)
df_enc = pd.get_dummies(df_ins, columns=["sex", "smoker", "region"],
                        drop_first=True, dtype=int)

X = df_enc.drop(columns=["charges"])
y = df_enc["charges"]

# Modelo (el mismo de la clase pasada, ahora con TODOS los datos)
modelo_ins = LinearRegression()
modelo_ins.fit(X, y)

print(f"Modelo cargado. R2: {modelo_ins.score(X, y):.4f}")

### Nueva librería: `statsmodels`

`sklearn` entrena modelos pero **no calcula p-values**. Para eso usamos `statsmodels`: una librería enfocada en el **análisis estadístico** de los modelos.

| | sklearn | statsmodels |
|---|---|---|
| **Enfoque** | Predecir | Entender |
| **API** | `fit / predict / score` | `OLS(y, X).fit()` |
| **Reporta p-values** | No | Sí |
| **Ideal para** | Producción, pipelines | Análisis, reportes |

Ambas calculan la **misma** regresión (mínimos cuadrados). Solo difieren en lo que **reportan**.

**Importante**: `statsmodels` necesita que agreguemos la columna del intercepto manualmente con `sm.add_constant()` (sklearn lo hace solo).

In [ ]:
import statsmodels.api as sm

# sklearn agrega el intercepto (beta_0) automaticamente.
# statsmodels NO lo hace, hay que pedirlo con add_constant.
# Lo que hace: agrega una columna de 1's al DataFrame.
X_con_intercepto = sm.add_constant(X)
X_con_intercepto.head(3)

La columna `const` (llena de 1's) es lo que permite que el modelo calcule el intercepto $\beta_0$.

Ahora ajustamos el modelo. `OLS` significa *Ordinary Least Squares* (mínimos cuadrados ordinarios) — **es exactamente la misma regresión** que `LinearRegression` de sklearn:

In [ ]:
# OLS = minimos cuadrados (lo mismo que LinearRegression de sklearn)
modelo_sm = sm.OLS(y, X_con_intercepto).fit()

# Los p-values estan en .pvalues
print(modelo_sm.pvalues.round(4))

### La matemática detrás del p-value: el estadístico t

Para cada coeficiente, `statsmodels` calcula un **estadístico t**:

$$t = \frac{\hat{\beta}}{\text{SE}(\hat{\beta})}$$

Donde:
- $\hat{\beta}$ es el coeficiente estimado (ej: +257 para `age`)
- $\text{SE}(\hat{\beta})$ es el **error estándar**: cuánto varía $\hat{\beta}$ si cambias la muestra. Mide la **incertidumbre**.

**Interpretación**: $t$ es el coeficiente dividido entre su incertidumbre.
- Si $|t|$ es **grande** (ej: 19.4 para `age`) → el coeficiente está **lejos de cero** → p-value bajo → **significativo**
- Si $|t|$ es **pequeño** (ej: -0.05 para `sex_male`) → el coeficiente está **cerca de cero** → p-value alto → **no significativo**

Veamos la tabla completa con coeficientes, t y p-values:

In [ ]:
# Tabla completa: coeficiente, error estandar, t, p-value
resumen = pd.DataFrame({
    "coeficiente": modelo_sm.params.round(1),
    "error_std":   modelo_sm.bse.round(1),
    "t":           modelo_sm.tvalues.round(2),
    "p_value":     modelo_sm.pvalues.round(4),
})

# Quitamos el intercepto (no nos interesa su p-value)
resumen = resumen.drop("const")

# Ordenamos por p-value (los mas significativos arriba)
resumen = resumen.sort_values("p_value")

# Marcamos cuales son significativas
resumen["significativa"] = "no"
resumen.loc[resumen["p_value"] < 0.05, "significativa"] = "SI"

resumen

### Lectura de la tabla

Las variables con p < 0.05 (**significativas**) son: `smoker_yes`, `age`, `bmi`, `children`. Puedes confiar en que sus coeficientes son reales.

Las variables con p > 0.05 son: `sex_male` (p=0.96!), `region_northwest` (p=0.48), `region_southeast` (p=0.22), `region_southwest` (p=0.12). **No podemos descartar que su efecto real sea cero**.

`sex_male` tiene p = 0.96 — eso significa que el coeficiente de -18 dólares es **completamente indistinguible de cero**. El sexo **no aporta nada** al modelo de seguros.

---

## 2. Selección de variables: ¿cuáles quito?

Tres criterios prácticos:

| Criterio | Cómo funciona |
|---|---|
| **P-value** | Si p > 0.05 → candidata a quitar |
| **Coeficiente estandarizado** | Si es ~0 comparado con los demás → poca importancia |
| **Comparar en test** | Entrena CON y SIN la variable. Si MAE no cambia → quítala |

Vamos a verificar con el criterio más práctico: **entrenar sin esas variables y comparar**.

In [ ]:
# Modelo COMPLETO (8 features)
pred_full = modelo_ins.predict(X)
mae_full = mean_absolute_error(y, pred_full)

# Modelo REDUCIDO: quitamos sex y las 3 regiones
cols_quitar = ["sex_male", "region_northwest", "region_southeast", "region_southwest"]
X_reducido = X.drop(columns=cols_quitar)
modelo_red = LinearRegression().fit(X_reducido, y)
mae_red = mean_absolute_error(y, modelo_red.predict(X_reducido))

print(f"MAE completo (8 features): ${mae_full:,.0f}")
print(f"MAE reducido (4 features): ${mae_red:,.0f}")
print(f"Diferencia:                ${abs(mae_full - mae_red):,.0f}")

---

## 3. Supuestos de la regresión lineal

Cuando usamos `LinearRegression`, el modelo **asume** ciertas cosas sobre los datos. Si no se cumplen, los resultados pueden ser incorrectos.

| Supuesto | Qué asume | Si se viola... |
|---|---|---|
| **Linealidad** | La relación entre X e y es una **línea recta** | La recta no captura la forma real → predicciones sesgadas |
| **Independencia** | Cada observación es independiente de las demás | Los errores estándar son incorrectos → p-values no confiables |
| **Homocedasticidad** | La varianza del error es **constante** (no crece ni decrece) | Las predicciones son más dudosas en ciertas zonas |
| **Normalidad de residuos** | Los residuos siguen distribución **Normal** | Los intervalos de confianza y p-values pierden validez |
| **No multicolinealidad** | Las features no están muy correlacionadas entre sí | Los coeficientes se vuelven inestables |

### ¿Cómo verifico si se cumplen?

La forma práctica: **graficar los residuos vs la predicción** y ver si hay patrones:

In [ ]:
# Residuos del modelo de insurance
y_pred_ins = modelo_ins.predict(X)
residuos_ins = y - y_pred_ins

sns.scatterplot(x=y_pred_ins, y=residuos_ins, alpha=0.5, color="#6B1525")
plt.axhline(0, color="#C82B40", linestyle="--")
plt.xlabel("Predicción")
plt.ylabel("Residuo (real - predicho)")
plt.title("Residuos vs Predicción (insurance)")
plt.show()

---

## 3.5 El patrón de sklearn: Crear → Fit → Predict → Evaluar

Antes de ver un modelo nuevo, fijemos el **patrón que sklearn usa para TODOS sus modelos**:

1. **Crear** el modelo: `modelo = LinearRegression()` (o `LogisticRegression()`, o `DecisionTree()`...)
2. **Entrenar** (fit): `modelo.fit(X_train, y_train)` — el modelo "aprende" los β a partir de los datos
3. **Predecir** (predict): `modelo.predict(X_test)` — aplica la fórmula aprendida a datos nuevos
4. **Evaluar**: comparar predicciones con la realidad (MAE, accuracy, etc.)

**Este flujo es SIEMPRE el mismo.** Lo que cambia es el modelo en el paso 1 y las métricas en el paso 4. Todo lo demás permanece idéntico.

Cuando veamos `LogisticRegression` en unos minutos, van a notar que **solo cambia UNA línea** del código.

### Cómo leer los residuos

| Lo que ves | Diagnóstico |
|---|---|
| **Nube aleatoria** alrededor de 0 | Todo bien. La regresión lineal es apropiada. |
| **Patrón curvo** (U o arco) | Relación no lineal. Necesitas un modelo más flexible. |
| **Abanico** (más dispersión a la derecha) | Heterocedasticidad. Las predicciones grandes son menos confiables. |
| **Bandas paralelas** | Hay un factor oculto que separa los datos en grupos. |

En nuestros residuos se ven **dos bandas claras** — eso es por los fumadores vs no fumadores. La regresión lineal funciona "bien" pero deja estructura sin capturar. Modelos más flexibles (Módulo 4) pueden mejorar esto.

---
---

## Bloque B — ¿Y si el target es SÍ/NO?

## 4. Nuevo dataset: Bank Marketing

In [ ]:
# Dataset de marketing bancario (desde el repo del curso)
URL_BANK = "https://raw.githubusercontent.com/cmosquerat/arca-diplomado/main/clase-14/bank.csv"
df = pd.read_csv(URL_BANK)

print(f"{df.shape[0]} clientes, {df.shape[1]} columnas")
df.head(3)

In [ ]:
# Veamos el target: response (0 = no acepta, 1 = si acepta)
print(df["response"].value_counts())
print()
print(f"Porcentaje que acepta: {df['response'].mean()*100:.1f}%")

**Solo el 11.7% acepta la campaña.** Esto es muy realista en marketing — la mayoría de personas dicen que no. El reto es identificar a ESE 11.7% antes de llamar.

### Pregunta de negocio

> *"¿A qué clientes debemos llamar para no desperdiciar tiempo ni presupuesto de marketing?"*

Eso es un problema de **clasificación binaria**: el target es 0/1 (no acepta / sí acepta).

---

## 5. LinearRegression para clasificar — funciona, pero...

Ya sabemos usar `LinearRegression`. El target es 0/1. **¿Podemos usarla tal cual?** Intentémoslo.

In [ ]:
# Preparamos los datos
# Seleccionamos columnas utiles (quitamos redundantes)
cols = ["age", "job", "marital", "education", "default", "balance",
        "housing", "loan", "contact", "duration", "campaign",
        "pdays", "previous", "poutcome", "response"]
df2 = df[cols].copy()

# Encoding (igual que en clase 12/13)
df_enc_bank = pd.get_dummies(df2,
    columns=["job", "marital", "education", "default",
             "housing", "loan", "contact", "poutcome"],
    drop_first=True, dtype=int)

X = df_enc_bank.drop(columns=["response"])
y = df_enc_bank["response"]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f"Train: {X_train.shape[0]} filas, Test: {X_test.shape[0]} filas")
print(f"Features: {X.shape[1]}")

In [ ]:
# Intentamos con LinearRegression (lo que ya sabemos)
modelo_lineal = LinearRegression()
modelo_lineal.fit(X_train, y_train)

# Veamos que predice
predicciones_raw = modelo_lineal.predict(X_test[:8])
print("Predicciones brutas (primeras 8 personas):")
for i, p in enumerate(predicciones_raw):
    print(f"  Cliente {i}: {p:.3f}")

**Problema**: las predicciones son cosas como -0.017, 0.196, 1.460...

- ¿Qué significa que un cliente acepte **-1.7%**?
- ¿O que acepte **146%**?

Esos valores **no son probabilidades válidas** (deberían estar entre 0 y 1). Pero... ¿y si usamos un truco simple?

### La idea del umbral

Si la predicción es **> 0.5**, decimos "sí". Si es **≤ 0.5**, decimos "no".

In [ ]:
# Aplicamos umbral: si prediccion > 0.5 -> 1, si no -> 0
y_pred_lineal = (modelo_lineal.predict(X_test) > 0.5).astype(int)

# Primeras 8 predicciones vs realidad
for i in range(8):
    raw = modelo_lineal.predict(X_test)[i]
    pred = y_pred_lineal[i]
    real = y_test.iloc[i]
    print(f"  Cliente {i}: raw={raw:.3f} -> pred={pred}, real={real}")

**Funciona.** Ahora tenemos predicciones 0/1. Pero ya no podemos usar MAE ni RMSE — necesitamos **métricas de clasificación**.

---

## 5.1 Nuevas métricas: accuracy

La más simple: ¿qué porcentaje acerté?

$$\text{Accuracy} = \frac{\text{predicciones correctas}}{\text{total}}$$

In [ ]:
acc = accuracy_score(y_test, y_pred_lineal)
print(f"Accuracy de LinearRegression + umbral: {acc:.4f} ({acc*100:.1f}%)")

**89.5% accuracy** — suena bien. Pero...

**Trampa**: solo el 11.7% de los clientes aceptan. Un modelo que **siempre diga "no"** tendría 88.3% de accuracy sin hacer nada. Nuestro modelo apenas supera a "siempre decir no".

**Accuracy sola no basta** cuando las clases están desbalanceadas. Necesitamos más detalle.

### 5.2 Matriz de confusión

En vez de un solo número, veamos **en qué se equivoca** el modelo:

In [ ]:
# Matriz de confusion
cm = confusion_matrix(y_test, y_pred_lineal)
print("Matriz de confusion:")
print(f"  Predicho:      NO    SI")
print(f"  Real NO:     {cm[0,0]:5d} {cm[0,1]:5d}")
print(f"  Real SI:     {cm[1,0]:5d} {cm[1,1]:5d}")

### Cómo leer esta matriz

|  | Predicho: NO | Predicho: SÍ |
|---|---|---|
| **Real: NO** | 7793 (acerté) | 159 (llamé innecesariamente) |
| **Real: SÍ** | 790 (**no lo llamé y hubiera aceptado**) | 301 (acerté) |

**El dato más grave**: de los 1091 clientes que **habrían dicho sí**, el modelo **solo detectó 301**. Se le escaparon **790 clientes potenciales**.

¿Qué es peor para el negocio?
- **Falso positivo** (llamar a alguien que no acepta): pierdes 5 minutos del call center
- **Falso negativo** (no llamar a alguien que sí habría aceptado): **pierdes un cliente**

### 5.3 Precision y recall

In [ ]:
# precision y recall
report = classification_report(y_test, y_pred_lineal, output_dict=True)
print(f"Precision (clase 1): {report['1']['precision']:.3f}")
print(f"Recall    (clase 1): {report['1']['recall']:.3f}")

- **Precision = 0.654**: "de los que dije SÍ, el 65% realmente eran SÍ". No está mal — cuando llamo, generalmente acierto.
- **Recall = 0.276**: "de los que realmente eran SÍ, **solo detecté el 28%**". Me escapan muchos clientes.

> **Hay tensión** entre precision y recall. Si bajo el umbral (de 0.5 a 0.3), detecto más clientes (sube recall) pero también llamo a más que no aceptan (baja precision). La decisión depende del **costo de cada error** para el negocio.

---

## 5.4 Los problemas de usar LinearRegression para clasificar

La regresión lineal "funciona" con el truco del umbral, pero tiene dos problemas de fondo:

1. **Las predicciones no son probabilidades**: un valor de -0.017 o 1.46 no se puede interpretar como "probabilidad de aceptar"
2. **No fue diseñada para esto**: OLS minimiza la suma de cuadrados, no la probabilidad de acertar la clase

Necesitamos un modelo que: (1) prediga valores entre 0 y 1 **siempre**, y (2) esté optimizado para clasificar.

**Eso es la regresión logística.**

---
---

## Bloque C — Regresión Logística

## 6. La idea: aplastamos la recta con una sigmoide

La regresión logística es regresión lineal + una función que "aplasta" el resultado al rango [0, 1].

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Donde $z = \beta_0 + \beta_1 x_1 + \ldots + \beta_n x_n$ (la misma combinación lineal de siempre).

Veamos qué forma tiene la sigmoide:

In [ ]:
# La funcion sigmoide: cualquier numero entra, sale un valor entre 0 y 1
z = np.linspace(-6, 6, 200)
sigma = 1 / (1 + np.exp(-z))

plt.plot(z, sigma, color="#C82B40", linewidth=2.5)
plt.axhline(0.5, color="gray", linestyle="--", alpha=0.5)
plt.axvline(0, color="gray", linestyle="--", alpha=0.5)
plt.xlabel("z (combinacion lineal)")
plt.ylabel("sigma(z) = probabilidad")
plt.title("La funcion sigmoide")
plt.show()

**Lo que la recta es a la regresión lineal, la sigmoide es a la logística.** Tiene forma de S:
- Si $z$ es muy negativo → $\sigma \approx 0$ (probabilidad baja)
- Si $z = 0$ → $\sigma = 0.5$ (50/50)
- Si $z$ es muy positivo → $\sigma \approx 1$ (probabilidad alta)

## 6.1 LogisticRegression en sklearn: una línea de diferencia

In [ ]:
# Estandarizamos (buena practica para logistica)
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

# UNA LINEA DE CAMBIO: LogisticRegression en vez de LinearRegression
modelo_log = LogisticRegression(max_iter=1000)
modelo_log.fit(X_train_s, y_train)

# Predecir (mismo .predict())
y_pred_log = modelo_log.predict(X_test_s)
print(f"Accuracy LogisticRegression: {accuracy_score(y_test, y_pred_log):.4f}")

### 6.2 Lo que ganas: `predict_proba()`

La logística no solo dice "sí o no". Da la **probabilidad** de cada clase:

In [ ]:
# predict_proba devuelve [P(no), P(si)] para cada cliente
probas = modelo_log.predict_proba(X_test_s[:8])

print("Cliente   P(no)    P(si)    Prediccion   Real")
print("-" * 55)
for i in range(8):
    pred = y_pred_log[i]
    real = y_test.iloc[i]
    print(f"   {i}      {probas[i,0]:.3f}    {probas[i,1]:.3f}       {pred}           {real}")

**Ahora SÍ puedes decir**: "este cliente tiene un 22.8% de probabilidad de aceptar". Eso es **imposible** con `LinearRegression`.

Esto es útil en la práctica porque puedes:
- Ordenar clientes por probabilidad → llamar primero a los más probables
- Ajustar el umbral según el costo de cada error

---

## 7. Comparación: Lineal vs Logística

In [ ]:
# Matrices de confusion lado a lado
cm_lin = confusion_matrix(y_test, y_pred_lineal)
cm_log = confusion_matrix(y_test, y_pred_log)

print("=== LinearRegression + umbral ===")
print(f"  Accuracy:  {accuracy_score(y_test, y_pred_lineal):.4f}")
print(f"  Precision: {cm_lin[1,1] / (cm_lin[0,1] + cm_lin[1,1]):.3f}")
print(f"  Recall:    {cm_lin[1,1] / (cm_lin[1,0] + cm_lin[1,1]):.3f}")
print(f"  VP: {cm_lin[1,1]}, FN: {cm_lin[1,0]}")
print()
print("=== LogisticRegression ===")
print(f"  Accuracy:  {accuracy_score(y_test, y_pred_log):.4f}")
print(f"  Precision: {cm_log[1,1] / (cm_log[0,1] + cm_log[1,1]):.3f}")
print(f"  Recall:    {cm_log[1,1] / (cm_log[1,0] + cm_log[1,1]):.3f}")
print(f"  VP: {cm_log[1,1]}, FN: {cm_log[1,0]}")

**Resumen**:
- Accuracy similar (~90% ambos)
- **Recall subió de 27.6% a 32.8%**: la logística detecta **57 clientes más** que habrían aceptado
- La logística da **probabilidades reales** (`predict_proba`)
- La logística fue **diseñada** para clasificar; la lineal solo "parchea" con un umbral

---

## 8. El flujo completo — otra vez

¿Recuerdan el flujo de la clase 13?

```
Split → Encode → Scale → Fit → Predict → Evaluar
```

**No cambió nada.** Solo reemplazamos `LinearRegression()` por `LogisticRegression()` y las métricas de regresión (MAE, RMSE, R²) por métricas de clasificación (accuracy, precision, recall, confusion matrix).

**Ese es el patrón de sklearn**: el modelo cambia, el flujo permanece.

---

## 9. Ejercicio

### Ajusta el umbral de la logística

En vez de usar `.predict()` (que usa 0.5 como umbral), usa `.predict_proba()` con un umbral de **0.3** para detectar más clientes:

In [ ]:
# Tu codigo aqui:
# 1. Obtener probabilidades con predict_proba
# 2. Aplicar umbral 0.3 (si P(si) > 0.3 -> predice 1)
# 3. Calcular accuracy, precision, recall
# 4. Comparar con el umbral de 0.5
# Pregunta: que paso con recall? y con precision?


---

## Resumen — herramientas de hoy

| Concepto | Código clave |
|---|---|
| P-values de coeficientes | `sm.OLS(y, X).fit().pvalues` |
| Selección de variables | Comparar MAE con/sin variables |
| Residuos (supuestos) | `sns.scatterplot(x=pred, y=residuos)` |
| Clasificación con lineal | `(modelo.predict(X) > 0.5).astype(int)` |
| Regresión logística | `LogisticRegression().fit(X, y)` |
| Probabilidades | `modelo.predict_proba(X)` |
| Accuracy | `accuracy_score(y_real, y_pred)` |
| Matriz de confusión | `confusion_matrix(y_real, y_pred)` |
| Precision / recall | `classification_report(y_real, y_pred)` |

## Cierre del Módulo 2

En 7 clases pasamos de describir datos a **predecir** con ellos:
- Target numérico → **regresión** → `LinearRegression`
- Target categórico → **clasificación** → `LogisticRegression`
- El flujo `split → encode → scale → fit → predict → evaluar` funciona para **todo**

**Próximo módulo (lunes 13 de abril)**: Módulo 3 — Análisis y Visualización de Datos. Ya saben modelar. Ahora van a aprender a **contar la historia** de los datos con gráficos.